In [6]:
import pandas as pd
from matplotlib.figure import Figure

df = pd.read_csv("../src_data/custom_filters.csv")
df

,runname,seed,steps,agg_score,commonsense_qa/acc,commonsense_qa/acc_norm,hellaswag/acc,hellaswag/acc_norm,openbookqa/acc,openbookqa/acc_norm,...,siqa/acc,siqa/acc_norm,winogrande/acc,winogrande/acc_norm,sciq/acc,sciq/acc_norm,arc/acc,arc/acc_norm,mmlu/acc,mmlu/acc_norm
0,filtering-baseline-2019-18-40gt,5,0,0.330953,0.186,0.233,0.272,0.258,0.166,0.286,...,0.367,0.362,0.516,0.497,0.210,0.202,0.2190,0.2515,0.230285,0.250127
1,filtering-baseline-2019-18-40gt,5,1000,0.357474,0.239,0.271,0.297,0.287,0.146,0.260,...,0.365,0.396,0.503,0.486,0.568,0.502,0.2665,0.2855,0.242526,0.253291
2,filtering-baseline-2019-18-40gt,5,2000,0.377436,0.280,0.284,0.321,0.332,0.134,0.268,...,0.368,0.399,0.519,0.502,0.686,0.590,0.3030,0.3215,0.245745,0.260988
3,filtering-baseline-2019-18-40gt,5,3000,0.387994,0.277,0.291,0.339,0.359,0.132,0.280,...,0.394,0.404,0.520,0.503,0.721,0.622,0.3210,0.3385,0.250427,0.264451
4,filtering-baseline-2019-18-40gt,5,4000,0.396110,0.299,0.315,0.340,0.366,0.158,0.286,...,0.376,0.399,0.515,0.500,0.739,0.620,0.3320,0.3445,0.256134,0.270382
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129,filtering-custom-short-line-ratio-0.67,6,10000,0.422300,0.333,0.341,0.382,0.417,0.192,0.318,...,0.389,0.407,0.536,0.530,NaN,NaN,0.3630,0.3700,0.266752,0.284400
130,filtering-custom-short-line-ratio-0.67,6,11000,0.425840,0.345,0.340,0.395,0.432,0.192,0.322,...,0.379,0.405,0.527,0.531,NaN,NaN,0.3680,0.3745,0.267998,0.282222
131,filtering-custom-short-line-ratio-0.67,6,12000,0.427343,0.339,0.348,0.397,0.439,0.198,0.316,...,0.382,0.402,0.535,0.536,NaN,NaN,0.3705,0.3795,0.268891,0.283246
132,filtering-custom-short-line-ratio-0.67,6,13000,0.429031,0.338,0.338,0.398,0.449,0.194,0.326,...,0.384,0.406,0.539,0.534,NaN,NaN,0.3655,0.3775,0.271709,0.282748


In [7]:
runs_mapping = {
    "filtering-baseline-2019-18-40gt": "Baseline",
    "filtering-custom-line-char-duplicated-v2-0.01": "Line duplicates filter",
    "filtering-custom-lines-punc-0.12": "Punctuation filter",
    "filtering-custom-short-line-ratio-0.67": "Short lines filter",
    "filtering-custom-punc0.12-short-lines0.67-line_char_dup0.1": "Filters combined",
}



In [11]:

from collections import defaultdict
import json
import os
from matplotlib import pyplot as plt
import orjson

metrics = ['agg_score', 'commonsense_qa/acc_norm', 'hellaswag/acc_norm', 'openbookqa/acc_norm', 'piqa/acc_norm',
                   'siqa/acc_norm', 'winogrande/acc_norm', 'arc/acc_norm', 'mmlu/acc_norm']

def normalize_runname(runname):
    return runname.replace("/", "_")

grouped = (
    df.groupby(["runname", "steps"])
    .agg(
        {
            key: "mean" for key in metrics
        }
    )
    .reset_index()
)

file_id="../assets/data/plots/custom_filters"
files = {}
for metric in metrics:
    datas = {}
    for name, group in grouped.groupby("runname"):
        group = group[["steps", metric]].sort_values(by="steps")
        group = group.set_index("steps")
        rolling_avg = group
        # rolling_avg = group.rolling(window=5).mean()
        datas[name] = {
            "x": (rolling_avg.index * 2048 * 1024 * 1e-9).tolist(),
            "y": rolling_avg[metric].tolist(),
            "label": runs_mapping[name],
        }
    # Sort the datata based on the steps
    datas = {k: v for k, v in sorted(datas.items(), key=lambda x: -x[1]["y"][-1])}
    # Create a folder
    os.makedirs(f"{file_id}", exist_ok=True)
    with open(f"{file_id}/{normalize_runname(metric)}.json", "w") as f:
        json.dump({
            "data": datas,
            "layout": {
                "title": {
                    "text": "Custom filters Performance"
                },
            }
        }, f)
    files[metric] = {"file": f"{normalize_runname(metric)}.json"}
# Create index
with open(f"{file_id}/index.json", "w") as f:
    json.dump({
        "files": files,
        "settings": {
            "defaultMetric": "agg_score",
            "slider":{"min":0,"max":10,"default":3}
        }
    }, f)
    